# Introduction
As we have completed a preprocessing pipeline, it is now time to select a model, perform feature 
tuning, and perform hyperparameter tunining. We will measure models on the following metrics:
- Precision (focused, it is more crucial to be correct than to flag all spam)
- Recall
- ROC Curve
- Accuracy

In [6]:
# use this cell to install all requirements for this project.
# !pip install -r requirements.txt

In [7]:
# imports
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

In [8]:
# make array print readable
np.set_printoptions(suppress=True, precision=4)

In [16]:
df = pd.read_csv(
    'data/cleaned_emails.csv', 
    index_col='filename', )
df.head()

,body,from,weekday,day,month,year,hour,timezone,to,subject,return_path,received,delivered_to,message_id,spam
filename,,,,,,,,,,,,,,,
01128.efb36914ecb55d78a894591eff0843c5,"['on', 'sun', 'NUMBER', 'jul', 'NUMBER', 'NUMB...",uni.de,sun,21,jul,2002,20,-400.0,freshrpms.net,"['re', 'ximian', 'apt', 'repo']",freshrpms.net,7,1,uni.de,False
00659.02e6dd777f837798533eae8f3b6a0491,"['what', 'is', 'mime', 'mime', 'stand', 'for',...",docserver.cac.washington.edu,mon,19,aug,2002,23,-700.0,example.sourceforge.net,"['wm', 'the', 'mime', 'inform', 'you', 'reques...",example.sourceforge.net,6,1,docserver.cac.washington.edu,False
00776.7df92458e9cf04b8873c406bde7d2fbe,"['im', 'not', 'up', 'to', 'fork', 'the', 'text...",golux.com,tue,13,aug,2002,15,-400.0,xent.com,"['a', 'messag', 'for', 'our', 'time']",xent.com,6,2,golux.com,False
00116.409b29c26edef06268b4bfa03ef1367a,"['on', 'sat', 'jul', 'NUMBER', 'NUMBER', 'at',...",skynet.ie,sat,20,jul,2002,13,100.0,linux.ie,"['re', 'ilug', 'vanquish', 'the', 'daemon', 'o...",linux.ie,8,1,skynet.ie,False
00615.23556d88fcb1179b25083cfc41017f42,"['origin', 'messag', 'date', 'thu', 'NUMBER', ...",dmv.com,thu,8,aug,2002,16,-400.0,example.sourceforge.net,"['re', 'razorus', 'use', 'razor', 'with', 'non...",example.sourceforge.net,7,1,landshark,False


In [18]:
# import feature pipeline
import cloudpickle as cp

feature_pipeline = None
with open('objects/feature_pipeline.pkl', 'rb') as f:
    feature_pipeline = cp.load(f)

testing = feature_pipeline.fit_transform(df)
mask = feature_pipeline.get_feature_names_out()

[Pipeline] ........ (step 1 of 3) Processing num_impute, total=   0.0s
[Pipeline]  (step 2 of 3) Processing create_path_length, total=   0.0s
[Pipeline] ......... (step 3 of 3) Processing num_scale, total=   0.0s
[ColumnTransformer] ........... (1 of 5) Processing num, total=   0.0s
[Pipeline] ........ (step 1 of 3) Processing cat_impute, total=   0.0s
[Pipeline] ....... (step 2 of 3) Processing cat_ordinal, total=   0.0s
[Pipeline] ...... (step 3 of 3) Processing cat_cyclical, total=   0.0s
[ColumnTransformer] ........... (2 of 5) Processing cat, total=   0.0s
[Pipeline] ........ (step 1 of 2) Processing day_impute, total=   0.0s
[Pipeline] ...... (step 2 of 2) Processing day_cyclical, total=   0.0s
[ColumnTransformer] ........... (3 of 5) Processing day, total=   0.0s
[Pipeline] .. (step 1 of 2) Processing text_create_list, total=   1.2s
[Pipeline] ....... (step 2 of 2) Processing text_sparse, total=   0.2s
[ColumnTransformer] .......... (4 of 5) Processing body, total=   1.4s
[Pipel

In [19]:
mask

array(['num__received', 'num__delivered_to', 'num__path_length',
       'cat__weekday_sin', 'cat__weekday_cos', 'cat__month_cos',
       'cat__month_sin', 'day__day_cos', 'day__day_sin', 'body__NUMBER',
       'body__URL', 'body__a', 'body__about', 'body__address',
       'body__all', 'body__also', 'body__an', 'body__and', 'body__ani',
       'body__are', 'body__as', 'body__at', 'body__be', 'body__been',
       'body__busi', 'body__but', 'body__by', 'body__can', 'body__click',
       'body__compani', 'body__do', 'body__dont', 'body__email',
       'body__file', 'body__for', 'body__free', 'body__from', 'body__get',
       'body__has', 'body__have', 'body__here', 'body__how', 'body__i',
       'body__if', 'body__in', 'body__inform', 'body__is', 'body__it',
       'body__just', 'body__like', 'body__list', 'body__mail',
       'body__make', 'body__me', 'body__messag', 'body__more', 'body__my',
       'body__need', 'body__new', 'body__no', 'body__not', 'body__now',
       'body__of', 'body_